# IMSDB Data Prep v2 — Genre Screenplay Dataset (No Scraping)

Uses `mocboch/movie_scripts` from HuggingFace Hub directly.
Filters by genre, cleans screenplay text, chunks into scenes,
formats into instruction-style samples, and pushes to HF Hub.

**Output datasets pushed to HF Hub:**
```
SatyaMoulika/imsdb-drama-screenplay
SatyaMoulika/imsdb-horror-screenplay
SatyaMoulika/imsdb-scifi-screenplay
SatyaMoulika/imsdb-comedy-screenplay
```

## 1 · Install Dependencies

In [ ]:
!pip install -q datasets huggingface_hub pandas tqdm

## 2 · HuggingFace Login

In [ ]:
from huggingface_hub import login, whoami
login()
print('Logged in as:', whoami()['name'])

## 3 · Config

In [ ]:
HF_USERNAME = 'SatyaMoulika'

# genre label in our pipeline -> column name in the dataset
GENRE_COL_MAP = {
    'Drama':   'Drama',
    'Horror':  'Horror',
    'Sci-Fi':  'Science Fiction',
    'Comedy':  'Comedy',
}

MIN_CHUNK_CHARS      = 300    # discard very short scenes
MAX_CHUNK_CHARS      = 2000   # discard very long scenes (context window safety)
TARGET_SAMPLES       = 800    # max samples per genre
VAL_RATIO            = 0.1

import os
DATA_DIR = './genre_data'
os.makedirs(DATA_DIR, exist_ok=True)

## 4 · Load Raw Dataset from HuggingFace

In [ ]:
import pandas as pd
from datasets import load_dataset

# Load the main CSV that contains scripts + genre columns
# data_files bypasses the viewer cast error caused by the metadata-only CSV
raw = load_dataset(
    'mocboch/movie_scripts',
    data_files='movie_scripts.csv',
    split='train',
)
df = raw.to_pandas()

print(f'Total rows: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nScript null count: {df["script"].isna().sum()}')
print(f'\nGenre counts:')
for label, col in GENRE_COL_MAP.items():
    print(f'  {label:10s} ({col}): {int(df[col].sum())} scripts')

## 5 · Filter by Genre + Drop Nulls

In [ ]:
genre_dfs = {}

for genre, col in GENRE_COL_MAP.items():
    subset = df[
        (df[col] == 1) &
        df['script'].notna() &
        (df['script'].str.strip() != '')
    ][['title', 'script', 'release_year', 'overview']].copy()

    subset['genre'] = genre
    genre_dfs[genre] = subset.reset_index(drop=True)
    print(f'{genre:10s} → {len(subset)} scripts available')

## 6 · Clean Raw Screenplay Text

In [ ]:
import re

def clean_screenplay(text: str) -> str:
    """Normalises whitespace and strips common IMSDB boilerplate."""
    # Remove HTML tags if any slipped through
    text = re.sub(r'<[^>]+>', '', text)
    # Normalise Windows line endings
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    # Collapse 3+ blank lines → 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Strip lone page numbers (e.g. "17." on its own line)
    text = re.sub(r'^\s*\d+\.?\s*$', '', text, flags=re.MULTILINE)
    # Strip revision markers
    text = re.sub(r'\(CONTINUED\)|CONTINUED:', '', text)
    # Strip "MORE" at end of page
    text = re.sub(r'^\s*\(MORE\)\s*$', '', text, flags=re.MULTILINE)
    return text.strip()


# Quick sanity check on one script
sample_script = genre_dfs['Drama']['script'].iloc[0]
cleaned        = clean_screenplay(sample_script)
print(f'Raw length:     {len(sample_script)}')
print(f'Cleaned length: {len(cleaned)}')
print('\nFirst 500 chars after cleaning:')
print(cleaned[:500])

## 7 · Chunk Screenplays into Scene-Level Samples

In [ ]:
SCENE_RE = re.compile(
    r'((?:INT|EXT|INT\.?\/EXT|I\/E)\.\s+[^\n]+)',
    re.IGNORECASE
)

def extract_scene_chunks(text: str, title: str, genre: str, overview: str) -> list[dict]:
    """
    Splits screenplay on INT./EXT. scene headings.
    Returns chunks within MIN/MAX_CHUNK_CHARS as training samples.
    """
    parts  = SCENE_RE.split(text)
    chunks = []

    i = 1
    while i < len(parts) - 1:
        heading = parts[i].strip()
        body    = parts[i + 1].strip() if i + 1 < len(parts) else ''
        chunk   = f'{heading}\n\n{body}'

        if MIN_CHUNK_CHARS <= len(chunk) <= MAX_CHUNK_CHARS:
            chunks.append({
                'title':         title,
                'genre':         genre,
                'overview':      overview or '',
                'scene_heading': heading,
                'screenplay':    chunk,
            })
        i += 2

    return chunks

## 8 · Build Per-Genre Chunk Collections

In [ ]:
from tqdm.auto import tqdm

genre_chunks = {genre: [] for genre in GENRE_COL_MAP}

for genre, gdf in genre_dfs.items():
    for _, row in tqdm(gdf.iterrows(), total=len(gdf), desc=genre):
        if len(genre_chunks[genre]) >= TARGET_SAMPLES:
            break

        cleaned = clean_screenplay(str(row['script']))
        chunks  = extract_scene_chunks(
            cleaned,
            title    = str(row['title']),
            genre    = genre,
            overview = str(row.get('overview', '')),
        )
        genre_chunks[genre].extend(chunks)

    # Trim to target
    genre_chunks[genre] = genre_chunks[genre][:TARGET_SAMPLES]
    print(f'{genre}: {len(genre_chunks[genre])} scene chunks collected')

## 9 · Format into Instruction-Style Training Samples

In [ ]:
def format_sample(sample: dict) -> dict:
    """
    Formats a scene chunk into the instruction template used during fine-tuning.

    This mirrors what the user will provide in the app:
      genre + scene heading + optional story context -> screenplay scene

    The `overview` field gives the model story-level context,
    which helps it produce genre-coherent content beyond just format mimicry.
    """
    overview_line = (
        f"Story context: {sample['overview']}"
        if sample['overview'] and sample['overview'] != 'nan'
        else f"Story context: From the {sample['genre'].lower()} film '{sample['title']}'"
    )

    instruction = (
        f"Write a {sample['genre']} screenplay scene.\n"
        f"Scene heading: {sample['scene_heading']}\n"
        f"{overview_line}"
    )

    text = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Screenplay:\n{sample['screenplay']}"
    )

    return {
        'instruction': instruction,
        'response':    sample['screenplay'],
        'text':        text,           # pre-formatted for Unsloth trainer
        'genre':       sample['genre'],
        'title':       sample['title'],
        'scene_heading': sample['scene_heading'],
    }


formatted = {
    genre: [format_sample(s) for s in chunks]
    for genre, chunks in genre_chunks.items()
}

print('Formatted sample counts:')
for genre, samples in formatted.items():
    print(f'  {genre}: {len(samples)}')

print('\n── Preview (Horror) ──')
print(formatted['Horror'][0]['text'][:600])

## 10 · Quality Checks

In [ ]:
import pandas as pd

for genre, samples in formatted.items():
    sdf = pd.DataFrame(samples)
    sdf['text_len']     = sdf['text'].str.len()
    sdf['response_len'] = sdf['response'].str.len()

    print(f'\n── {genre} ──')
    print(sdf[['text_len', 'response_len']].describe()[['min', 'mean', 'max']].to_string())

    # Check for duplicate scenes (same heading + same title)
    dupes = sdf.duplicated(subset=['title', 'scene_heading']).sum()
    if dupes:
        print(f'  Duplicate scenes: {dupes} — dropping')
        sdf = sdf.drop_duplicates(subset=['title', 'scene_heading'])
        formatted[genre] = sdf.to_dict('records')

    # Verify no empty responses
    empty = sdf[sdf['response_len'] < MIN_CHUNK_CHARS]
    if len(empty):
        print(f'  Short responses: {len(empty)} — dropping')
        formatted[genre] = sdf[sdf['response_len'] >= MIN_CHUNK_CHARS].to_dict('records')

## 11 · Train / Validation Split

In [ ]:
import random
from datasets import Dataset, DatasetDict

random.seed(42)
genre_datasets = {}

for genre, samples in formatted.items():
    random.shuffle(samples)
    split = int(len(samples) * (1 - VAL_RATIO))

    genre_datasets[genre] = DatasetDict({
        'train':      Dataset.from_list(samples[:split]),
        'validation': Dataset.from_list(samples[split:]),
    })
    print(f'{genre}: {split} train / {len(samples)-split} val')

## 12 · Save Locally as JSONL

In [ ]:
import json

for genre, ds in genre_datasets.items():
    slug = genre.lower().replace('-', '_').replace(' ', '_').replace('/', '_')
    for split in ['train', 'validation']:
        path = f'{DATA_DIR}/{slug}_{split}.jsonl'
        with open(path, 'w') as f:
            for row in ds[split]:
                f.write(json.dumps(row) + '\n')
        print(f'Saved {path}  ({len(ds[split])} rows)')

## 13 · Push to HuggingFace Hub

In [ ]:
for genre, ds in genre_datasets.items():
    slug    = genre.lower().replace('-', '_').replace('/', '_')
    repo_id = f'{HF_USERNAME}/imsdb-{slug}-screenplay'

    ds.push_to_hub(repo_id, private=False)
    print(f'Pushed → https://huggingface.co/datasets/{repo_id}')

## 14 · Final Summary

In [ ]:
print('='*55)
print('DATASET SUMMARY')
print('='*55)
for genre, ds in genre_datasets.items():
    slug    = genre.lower().replace('-', '_').replace('/', '_')
    repo_id = f'{HF_USERNAME}/imsdb-{slug}-screenplay'
    total   = len(ds['train']) + len(ds['validation'])
    print(f"""
{genre}
  Total samples : {total} ({len(ds['train'])} train / {len(ds['validation'])} val)
  Chunk size    : {MIN_CHUNK_CHARS}–{MAX_CHUNK_CHARS} chars per scene
  Format        : ### Instruction / ### Screenplay
  HF Repo       : https://huggingface.co/datasets/{repo_id}
""")